# Traffic Signal Optimization with RL

Everything runs from this notebook. No terminal commands needed.

## If a cell dies, just re-run it

Long training cells can be killed by a kernel restart, a VS Code disconnect,
or the laptop sleeping. Nothing is lost when that happens:

| What | On re-run |
| --- | --- |
| Training | Resumes from the last checkpoint (saved every 20k steps) |
| Baselines | Reads the cache — completed scenarios are skipped |
| Evaluation | Reads the cache — completed models are skipped |

Re-running a cell that already finished takes seconds. So the safe move after
any interruption is simply: run the cell again.

## Watching progress

Training prints one line per job, not a live progress bar (tqdm output over
25 long runs bloats the notebook file and is a common way to hang the kernel).
For live learning curves, run TensorBoard from the VS Code terminal:

    tensorboard --logdir tb/

or just check back on the printed timings.

## Setup

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

from ppo import train as train_ppo
from dqn import train as train_dqn
from ippo import train as train_ippo
import fixed_time
import max_pressure
from utils.evaluate import evaluate, evaluate_marl

from stable_baselines3 import PPO, DQN

# ── Configuration ────────────────────────────────────────────────────
TRAINING_SEEDS = [42, 43, 44, 45, 46]      # PPO vs DQN comparison
ABLATION_SEEDS = [42, 43, 44]               # reward ablation
MARL_SEEDS     = [42, 43, 44]               # cologne3 / cologne8
EVAL_SEEDS     = [100, 200, 300, 400, 500]  # held-out evaluation
ALT_REWARDS    = ["pressure", "queue", "average-speed"]
MARL_CONFIG    = {"cologne3": 500_000, "cologne8": 800_000}

CACHE = Path("results/cache")
CACHE.mkdir(parents=True, exist_ok=True)


def cached(name, compute):
    """Read results/cache/<name>.csv if it exists, else compute and save it.

    This is what makes the notebook restartable: an interrupted cell loses
    only the job it was in the middle of, not everything before it.
    """
    path = CACHE / f"{name}.csv"
    if path.exists():
        print(f"  cached: {name}")
        return pd.read_csv(path)
    df = compute()
    df.to_csv(path, index=False)
    print(f"  saved:  {name}")
    return df


print("Setup OK")

---
## Phase 1 — Baselines

Fixed-time and max-pressure on all three scenarios. No training, but cologne8
takes a while, so each (scenario, algo) pair is cached separately.

In [ ]:
baseline_frames = []
for scenario in ["cologne1", "cologne3", "cologne8"]:
    for name, runner in [("fixed_time", fixed_time.run),
                         ("max_pressure", max_pressure.run)]:
        df = cached(
            f"baseline_{scenario}_{name}",
            lambda r=runner, s=scenario: r(scenario=s, eval_seeds=EVAL_SEEDS),
        )
        df["scenario"] = scenario
        df["algo"] = name
        baseline_frames.append(df)

baselines = pd.concat(baseline_frames, ignore_index=True)
baselines.to_csv("results/baselines.csv", index=False)

baselines.groupby(["scenario", "algo"]).agg(
    wait=("mean_wait", "mean"),
    speed=("mean_speed", "mean"),
    stopped=("mean_stopped", "mean"),
).round(2)

---
## Phase 2 — Single-intersection RL (cologne1)

Training is split across several cells so no single cell runs for hours.
Each prints one line per job with its runtime.

### 2.1 PPO — 5 seeds

~30 min per seed on WSL2 + libsumo, ~4h on Windows + TraCI.

In [ ]:
for seed in TRAINING_SEEDS:
    train_ppo(seed=seed)

### 2.2 DQN — 5 seeds

In [ ]:
for seed in TRAINING_SEEDS:
    train_dqn(seed=seed)

### 2.3 Reward ablation — PPO, 3 alternative rewards x 3 seeds

Nine runs. Run these one reward at a time if you want shorter cells — just
change `ALT_REWARDS` to a single-item list and re-run.

In [ ]:
for reward in ALT_REWARDS:
    for seed in ABLATION_SEEDS:
        train_ppo(seed=seed, reward_fn=reward)

### 2.4 Evaluate

Models that aren't trained yet are skipped with a message, so you can run this
at any point to see partial results.

In [ ]:
def eval_one(algo, seed, reward_fn="diff-waiting-time"):
    """Evaluate one trained cologne1 model, using the cache if available."""
    tag = f"seed{seed}"
    if reward_fn != "diff-waiting-time":
        tag += f"_{reward_fn}"

    def compute():
        model_path = Path(f"checkpoints/cologne1/{algo}/{tag}/final.zip")
        if not model_path.exists():
            raise FileNotFoundError(f"not trained yet: {model_path}")
        cls = PPO if algo == "ppo" else DQN
        model = cls.load(str(model_path))
        return evaluate(model, eval_seeds=EVAL_SEEDS,
                        reward_fn=reward_fn, scenario="cologne1")

    try:
        df = cached(f"eval_cologne1_{algo}_{tag}", compute)
    except FileNotFoundError as e:
        print(f"  skip: {e}")
        return None

    df["algo"] = algo
    df["train_seed"] = seed
    df["reward_fn"] = reward_fn
    df["scenario"] = "cologne1"
    return df


frames = []
for seed in TRAINING_SEEDS:
    for algo in ["ppo", "dqn"]:
        df = eval_one(algo, seed)
        if df is not None:
            frames.append(df)

for reward in ALT_REWARDS:
    for seed in ABLATION_SEEDS:
        df = eval_one("ppo", seed, reward_fn=reward)
        if df is not None:
            frames.append(df)

cologne1_results = pd.concat(frames, ignore_index=True)
cologne1_results.to_csv("results/cologne1_rl.csv", index=False)
print(f"\n{len(cologne1_results)} rows -> results/cologne1_rl.csv")

### 2.5 Summary table

In [ ]:
cologne1_summary = cologne1_results.groupby(["algo", "reward_fn"]).agg(
    n_seeds=("train_seed", "nunique"),
    wait_mean=("mean_wait", "mean"),
    wait_std=("mean_wait", "std"),
    speed_mean=("mean_speed", "mean"),
    speed_std=("mean_speed", "std"),
    stopped_mean=("mean_stopped", "mean"),
    stopped_std=("mean_stopped", "std"),
).round(3)
cologne1_summary

### 2.6 PPO vs DQN

In [ ]:
comp = cologne1_results[cologne1_results["reward_fn"] == "diff-waiting-time"]
agg = comp.groupby("algo").agg(
    wait_mean=("mean_wait", "mean"), wait_std=("mean_wait", "std"),
    speed_mean=("mean_speed", "mean"), speed_std=("mean_speed", "std"),
    stopped_mean=("mean_stopped", "mean"), stopped_std=("mean_stopped", "std"),
).reset_index()

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
colors = {"ppo": "#0ea5e9", "dqn": "#e8772e"}
for ax, (m, s, label) in zip(axes, [
    ("wait_mean", "wait_std", "Mean wait (s)"),
    ("speed_mean", "speed_std", "Mean speed (m/s)"),
    ("stopped_mean", "stopped_std", "Avg stopped vehicles"),
]):
    ax.bar(agg["algo"].str.upper(), agg[m], yerr=agg[s],
           color=[colors[a] for a in agg["algo"]], capsize=8,
           edgecolor="black", linewidth=0.5)
    for i, (mi, si) in enumerate(zip(agg[m], agg[s])):
        ax.text(i, mi + si + max(agg[m]) * 0.03, f"{mi:.2f}",
                ha="center", va="bottom", fontsize=10)
    ax.set_ylabel(label)
    ax.grid(axis="y", alpha=0.3)
fig.suptitle("PPO vs DQN on cologne1 (5 training seeds, mean +/- std)", fontweight="bold")
fig.tight_layout()
plt.savefig("results/cologne1_ppo_vs_dqn.png", dpi=150, bbox_inches="tight")
plt.show()

### 2.7 Reward ablation

In [ ]:
ppo_only = cologne1_results[cologne1_results["algo"] == "ppo"]
abl = ppo_only.groupby("reward_fn").agg(
    wait_mean=("mean_wait", "mean"), wait_std=("mean_wait", "std"),
    p95_mean=("p95_wait", "mean"),   p95_std=("p95_wait", "std"),
).reset_index().sort_values("wait_mean")

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for ax, (m, s, label) in zip(axes, [
    ("wait_mean", "wait_std", "Mean waiting time (s) - lower is better"),
    ("p95_mean",  "p95_std",  "p95 waiting time (s) - tail / fairness"),
]):
    ax.bar(abl["reward_fn"], abl[m], yerr=abl[s], color="#0ea5e9",
           capsize=8, edgecolor="black", linewidth=0.5)
    for i, (mi, si) in enumerate(zip(abl[m], abl[s])):
        ax.text(i, mi + si + max(abl[m]) * 0.03, f"{mi:.2f}",
                ha="center", va="bottom", fontsize=10)
    ax.set_ylabel(label)
    ax.set_xticklabels(abl["reward_fn"], rotation=15, ha="right")
    ax.grid(axis="y", alpha=0.3)
fig.suptitle("PPO reward-function ablation on cologne1", fontweight="bold")
fig.tight_layout()
plt.savefig("results/cologne1_reward_ablation.png", dpi=150, bbox_inches="tight")
plt.show()

---
## Phase 3 — Multi-intersection MARL

Shared-policy IPPO. With N intersections, SB3 counts each env step as N
timesteps, so cologne8 needs more timesteps for comparable training depth.

### 3.1 cologne3 — 3 signals, 3 seeds

In [ ]:
for seed in MARL_SEEDS:
    train_ippo(seed=seed, scenario="cologne3", timesteps=MARL_CONFIG["cologne3"])

### 3.2 cologne8 — 8 signals, 3 seeds

The longest cell in the notebook (~2.5h per seed on WSL2, much longer on
native Windows). If it gets interrupted, re-run — it resumes.

In [ ]:
for seed in MARL_SEEDS:
    train_ippo(seed=seed, scenario="cologne8", timesteps=MARL_CONFIG["cologne8"])

### 3.3 Evaluate

In [ ]:
def eval_marl_one(scenario, seed):
    def compute():
        model_path = Path(f"checkpoints/{scenario}/ippo/seed{seed}/final.zip")
        if not model_path.exists():
            raise FileNotFoundError(f"not trained yet: {model_path}")
        model = PPO.load(str(model_path))
        return evaluate_marl(model, eval_seeds=EVAL_SEEDS, scenario=scenario)

    try:
        df = cached(f"eval_{scenario}_ippo_seed{seed}", compute)
    except FileNotFoundError as e:
        print(f"  skip: {e}")
        return None

    df["algo"] = "ippo"
    df["train_seed"] = seed
    df["scenario"] = scenario
    return df


marl_frames = []
for scenario in MARL_CONFIG:
    for seed in MARL_SEEDS:
        df = eval_marl_one(scenario, seed)
        if df is not None:
            marl_frames.append(df)

marl_results = pd.concat(marl_frames, ignore_index=True)
marl_results.to_csv("results/marl.csv", index=False)
print(f"\n{len(marl_results)} rows -> results/marl.csv")

### 3.4 IPPO vs baselines

In [ ]:
marl_scenarios = list(MARL_CONFIG.keys())
b = baselines[baselines["scenario"].isin(marl_scenarios)][
    ["scenario", "algo", "mean_wait", "mean_speed", "mean_stopped", "p95_wait"]
].copy()
b["train_seed"] = -1
m = marl_results[
    ["scenario", "algo", "train_seed", "mean_wait", "mean_speed", "mean_stopped", "p95_wait"]
].copy()
combined = pd.concat([b, m], ignore_index=True)

agg = combined.groupby(["scenario", "algo"]).agg(
    wait_mean=("mean_wait", "mean"), wait_std=("mean_wait", "std"),
).reset_index()

fig, ax = plt.subplots(figsize=(10, 5))
scenarios = sorted(agg["scenario"].unique())
algos = ["fixed_time", "max_pressure", "ippo"]
colors = {"fixed_time": "#94a3b8", "max_pressure": "#a855f7", "ippo": "#0ea5e9"}

x = np.arange(len(scenarios))
width = 0.27
for i, algo in enumerate(algos):
    rows = agg[agg["algo"] == algo].set_index("scenario").reindex(scenarios).reset_index()
    means = rows["wait_mean"].fillna(0).values
    stds = rows["wait_std"].fillna(0).values
    ax.bar(x + (i - 1) * width, means, width, yerr=stds, label=algo,
           color=colors[algo], capsize=4, edgecolor="black", linewidth=0.5)

ax.set_xticks(x)
ax.set_xticklabels(scenarios)
ax.set_ylabel("Mean waiting time (s)")
ax.set_title("Multi-intersection: IPPO vs classical baselines", fontweight="bold")
ax.legend()
ax.grid(axis="y", alpha=0.3)
fig.tight_layout()
plt.savefig("results/marl_vs_baselines.png", dpi=150, bbox_inches="tight")
plt.show()
agg

---
## Final combined table

In [ ]:
b = baselines.copy()
b["train_seed"] = -1
b["reward_fn"] = "n/a"

c1 = cologne1_results[cologne1_results["reward_fn"] == "diff-waiting-time"].copy()
cols = ["scenario", "algo", "train_seed", "reward_fn",
        "mean_wait", "mean_speed", "mean_stopped", "p95_wait"]

final = pd.concat([
    b[cols],
    c1[cols],
    marl_results.assign(reward_fn="diff-waiting-time")[cols],
], ignore_index=True)

final_summary = final.groupby(["scenario", "algo"]).agg(
    n_seeds=("train_seed", "nunique"),
    wait_mean=("mean_wait", "mean"),  wait_std=("mean_wait", "std"),
    speed_mean=("mean_speed", "mean"), speed_std=("mean_speed", "std"),
    stopped_mean=("mean_stopped", "mean"), stopped_std=("mean_stopped", "std"),
    p95_mean=("p95_wait", "mean"), p95_std=("p95_wait", "std"),
).round(3)

final_summary.to_csv("results/final_summary.csv")
print("Saved -> results/final_summary.csv")
final_summary

---
## Utility — check what's done

Run this any time to see which jobs are finished, in progress, or not started.

In [ ]:
from utils.resume import latest_checkpoint

rows = []
jobs = (
    [("ppo", "cologne1", f"seed{s}", 200_000) for s in TRAINING_SEEDS]
    + [("dqn", "cologne1", f"seed{s}", 200_000) for s in TRAINING_SEEDS]
    + [("ppo", "cologne1", f"seed{s}_{r}", 200_000)
       for r in ALT_REWARDS for s in ABLATION_SEEDS]
    + [("ippo", sc, f"seed{s}", t)
       for sc, t in MARL_CONFIG.items() for s in MARL_SEEDS]
)

for algo, scenario, tag, target in jobs:
    d = Path(f"checkpoints/{scenario}/{algo}/{tag}")
    if (d / "final.zip").exists():
        status, pct = "done", 100.0
    else:
        _, steps = latest_checkpoint(d, algo)
        status = "partial" if steps else "not started"
        pct = 100.0 * steps / target
    rows.append({"scenario": scenario, "algo": algo, "run": tag,
                 "status": status, "percent": round(pct, 1)})

status_df = pd.DataFrame(rows)
n_done = (status_df["status"] == "done").sum()
print(f"{n_done}/{len(status_df)} training jobs complete\n")
status_df